# Experiment: SepAware strengthened two-dataset JMIR experiment

Objective:
- Test whether separability-aware post-generation selection improves rare-outcome prediction beyond class-frequency correction.
- Require held-out-real AUPRC improvement without material deterioration in minority coverage, calibration, or the membership-risk proxy.

This notebook is the readable entry point. The versioned experiment implementation is in `JMIR_SepAware/experiments/run_strengthened_experiments.py`, and the prospective protocol is in `JMIR_SepAware/experiments/PROTOCOL.md`.


In [1]:
# Setup: imports and reproducibility
from __future__ import annotations

import json
import subprocess
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'JMIR_SepAware').exists():
    candidate = ROOT.parents[1]
    if (candidate / 'JMIR_SepAware').exists():
        ROOT = candidate
    else:
        raise RuntimeError('Could not locate the Qualification Experiment workspace root.')
SEED = 42
OUTPUT = ROOT / 'JMIR_SepAware' / 'experiments' / 'outputs'
{'root': str(ROOT), 'seed': SEED, 'output': str(OUTPUT)}


{'root': '/Users/User/Downloads/Qualification Experiment',
 'seed': 42,
 'output': '/Users/User/Downloads/Qualification Experiment/JMIR_SepAware/experiments/outputs'}

## Plan

- **Primary hypothesis:** SepAware improves held-out-real AUPRC relative to the corresponding standard generator.
- **Boundary condition:** the effect varies with intrinsic class overlap and dataset.
- **Methods:** real only, random over/undersampling, SMOTE, class weighting, CTGAN, TVAE, and SepAware variants.
- **Models:** logistic regression, random forest, and CatBoost.
- **Metrics:** AUPRC, Macro-F1, minority F1/precision/recall, AUROC, balanced accuracy, Brier score, calibration slope/intercept, minority coverage/diversity, exact matches, and membership proxy AUROC.


In [2]:
# Set MODE='smoke' for a quick integrity test or MODE='full' for the manuscript run.
MODE = 'smoke'
command = [
    str(ROOT / '.venv-jmir' / 'bin' / 'python'),
    str(ROOT / 'JMIR_SepAware' / 'experiments' / 'run_strengthened_experiments.py'),
    '--mode', MODE,
]
command


['/Users/User/Downloads/Qualification Experiment/.venv-jmir/bin/python',
 '/Users/User/Downloads/Qualification Experiment/JMIR_SepAware/experiments/run_strengthened_experiments.py',
 '--mode',
 'smoke']

## Execute

The script writes fold-level checkpoints after every fold. Generator runs may take substantial time.


In [3]:
# Uncomment to execute from the notebook.
# completed = subprocess.run(command, cwd=ROOT, check=True, text=True)
print('Ready:', ' '.join(map(str, command)))


Ready: /Users/User/Downloads/Qualification Experiment/.venv-jmir/bin/python /Users/User/Downloads/Qualification Experiment/JMIR_SepAware/experiments/run_strengthened_experiments.py --mode smoke


## Results and audit

Load the tidy exports only after the run completes. Keep TSTR qualification results separate from the augmentation results produced here.


In [4]:
summary_path = OUTPUT / 'predictive_metrics_summary.csv'
diagnostic_path = OUTPUT / 'synthetic_diagnostics_long.csv'
if summary_path.exists():
    display(pd.read_csv(summary_path).sort_values(['dataset', 'classifier', 'auprc_mean'], ascending=[True, True, False]))
else:
    print('No completed summary yet.')
if diagnostic_path.exists():
    display(pd.read_csv(diagnostic_path).groupby(['dataset', 'condition']).mean(numeric_only=True))


,dataset,condition,classifier,n,auprc_mean,auprc_sd,macro_f1_mean,macro_f1_sd,minority_f1_mean,minority_f1_sd,minority_recall_mean,minority_recall_sd,auroc_mean,auroc_sd,brier_mean,brier_sd
9,COVID-19,Real only,CatBoost,15,0.355901,0.056848,0.514452,0.059055,0.205711,0.106244,0.155882,0.091034,0.596101,0.061595,0.202757,0.013642
15,COVID-19,SepAware CTGAN,CatBoost,15,0.352722,0.074100,0.534660,0.074984,0.321248,0.131026,0.365441,0.172525,0.592701,0.064247,0.235074,0.024734
21,COVID-19,Standard CTGAN,CatBoost,15,0.349944,0.088348,0.512975,0.049329,0.285881,0.075952,0.309804,0.102438,0.569497,0.090901,0.233091,0.029920
0,COVID-19,Class weighting,CatBoost,15,0.341272,0.068537,0.536075,0.067644,0.328641,0.097645,0.364461,0.112512,0.574739,0.060217,0.235463,0.025403
3,COVID-19,Random oversampling,CatBoost,15,0.340147,0.066250,0.538954,0.087748,0.319080,0.122203,0.328676,0.120923,0.578894,0.061848,0.229694,0.024486
6,COVID-19,Random undersampling,CatBoost,15,0.331731,0.063397,0.512338,0.059286,0.394005,0.067582,0.597549,0.110407,0.584583,0.084322,0.301065,0.044490
24,COVID-19,Standard TVAE,CatBoost,15,0.329110,0.056359,0.527933,0.068996,0.262653,0.121489,0.241667,0.125612,0.587676,0.045690,0.224864,0.020798
18,COVID-19,SepAware TVAE,CatBoost,15,0.321585,0.036607,0.490774,0.058336,0.192486,0.100009,0.164461,0.095754,0.575482,0.047323,0.226730,0.017619
12,COVID-19,SMOTE,CatBoost,15,0.289833,0.044173,0.491769,0.049141,0.221044,0.076012,0.209559,0.080763,0.544105,0.051420,0.233436,0.022050
4,COVID-19,Random oversampling,LogisticRegression,15,0.398528,0.069432,0.538814,0.040287,0.380959,0.057331,0.506618,0.107772,0.625025,0.060048,0.245882,0.021723


repeat  fold  generator_seed  generator_seconds  \
dataset  condition                                                         
COVID-19 SepAware CTGAN     1.0   2.0         11116.0           1.411317   
         SepAware TVAE      1.0   2.0         21116.0           0.401081   
         Standard CTGAN     1.0   2.0         11116.0           1.406316   
         Standard TVAE      1.0   2.0         21116.0           0.396352   
Vigitel  SepAware CTGAN     1.0   2.0         11116.0           8.869564   
         SepAware TVAE      1.0   2.0         21116.0           1.717488   
         Standard CTGAN     1.0   2.0         11116.0           8.684673   
         Standard TVAE      1.0   2.0         21116.0           1.644365   

                         minority_coverage_distance  minority_diversity  \
dataset  condition                                                        
COVID-19 SepAware CTGAN                    4.301453            7.166733   
         SepAware TVAE                     6.038456            0.604790   
         Standard CTGAN                    4.437124           11.119400   
         Standard TVAE                     5.883417            1.011150   
Vigitel  SepAware CTGAN                    1.714054            4.334541   
         SepAware TVAE                     4.184677            0.227226   
         Standard CTGAN                    1.986678            5.554847   
         Standard TVAE                     3.898769            1.343015   

                         exact_match_rate  membership_auc  
dataset  condition                                         
COVID-19 SepAware CTGAN               0.0        0.514216  
         SepAware TVAE                0.0        0.512292  
         Standard CTGAN               0.0        0.517108  
         Standard TVAE                0.0        0.515629  
Vigitel  SepAware CTGAN               0.0        0.513588  
         SepAware TVAE                0.0        0.500230  
         Standard CTGAN               0.0        0.502220  
         Standard TVAE                0.0        0.500403

## Interpretation gate

- Compare SepAware with the matching standard generator using paired fold/seed differences.
- Do not call a method beneficial from Macro-F1 alone.
- Inspect calibration, coverage, diversity, and membership risk before drawing a conclusion.
- Record generator collapse or conditional-sampling failures as results, not as runs to hide.
